# Validate and route to the dead-letter path

A validator that returns `True`/`False` gives you a dead-letter queue
full of rows and a metric that cannot be split by cause. A validator
that returns a **reason code** gives you a metric label, a partition
key and a routing decision.

This notebook runs the malformed dataset through the validator and
shows the complete failure taxonomy.

## 0. Setup

Put the course's pure core modules on the path. No installation
and no cloud SDK — that is the whole point of the split.

In [ ]:
import sys
from pathlib import Path

COURSE_ROOT = Path.cwd()
if not (COURSE_ROOT / "project" / "core").exists():
    COURSE_ROOT = Path.cwd().parent
sys.path.insert(0, str(COURSE_ROOT / "project" / "core"))

SAMPLE_DIR = COURSE_ROOT / "sample_data"


## 1. The malformed file

Twelve rows, each one malformed in a different way. The file is
deliberately exhaustive: every reason the validator can emit
appears at least once, so the taxonomy is testable rather than a
hope.

In [ ]:
from pipeline_logic import ingest_events, reject_counts

malformed = (SAMPLE_DIR / 'malformed_events.csv').read_text(encoding='utf-8').splitlines()
print(f'malformed lines: {len(malformed)}')
for line in malformed:
    print(line)

## 2. The rejection reasons

Each row is rejected with a stable, low-cardinality code. This is
what you group by when you build the metric.

In [ ]:
accepted, rejected = ingest_events(malformed)
print(f'accepted: {len(accepted)}')
for reason, count in sorted(reject_counts(rejected).items()):
    print(f'{reason:<24} {count}')

## 3. The detail of each rejection

A reason code on its own tells you *what* failed. The detail tells
you *why* — and it is the only part you must not put in a metric
label, because it is unbounded.

In [ ]:
for item in rejected:
    print(item['reason'].ljust(24) + ' ' + item['detail'])

## 4. Why a reason code and not an exception

If the validator raised, the first bad row would kill the batch.
If it returned `False`, you would know *that* a row failed but not
*why*. The reason code gives you all three: the row survives, the
cause is recorded, and the metric is splittable.

In [ ]:
from pipeline_logic import validate_event

sample = {
    'event_id': 'e1',
    'user_id': 'u1',
    'event_ts': '2026-01-15T00:00:00+00:00',
    'event_type': 'order',
    'region': 'EMEA',
    'channel': 'web',
    'currency': 'EUR',
    'amount': 10.0,
}
print(validate_event(sample))

broken = dict(sample, region='MARS')
print(validate_event(broken))